# Our Approch



Our approach is based on finding the critical points of the loss function and then determining which of these points correspond to minima using the first- or second-derivative test. Rather than solving the optimization problem through the normal equations used in linear regression, we analyze the derivative of the loss with respect to the model parameters directly.

For a model of the form

$$
f(X)=\sigma(XW+b),
$$

the parameters \(W\) and \(b\) are optimized by minimizing a loss function \(L(W,b)\). A necessary condition for an interior local minimum is that the gradient of the loss vanishes:

$$
\nabla_{W,b}L(W,b)=0.
$$

Thus, our first step is to find the **critical points** of the loss function by solving the corresponding derivative equations. Once the critical points have been obtained, we determine their nature using the first- or second-derivative test. In one dimension, a critical point \(x^*\) can be classified using the second derivative:

$$
L''(x^*)>0
\quad\Longrightarrow\quad
x^* \text{ is a local minimum},
$$

while

$$
L''(x^*)<0
\quad\Longrightarrow\quad
x^* \text{ is a local maximum}.
$$

In multiple dimensions, the analogous test is performed using the Hessian matrix. At a critical point \(w^*\), if the Hessian is positive definite, then \(w^*\) is a strict local minimum:

$$
\nabla L(w^*)=0,
\qquad
\nabla^2L(w^*)\succ0.
$$

This approach requires the loss function to have derivatives that can be explicitly characterized. This is one of the reasons the choice of activation function is important in our analysis.

### Why Not ReLU or Tanh?

The ReLU activation function is defined as

$$
\operatorname{ReLU}(z)=\max(0,z).
$$

Although ReLU is differentiable for \(z\neq0\), it is not differentiable at \(z=0\). Its derivative is

$$
\operatorname{ReLU}'(z)=
\begin{cases}
0, & z<0,\\
1, & z>0.
\end{cases}
$$

Consequently, a loss function involving ReLU have infinite critical points where the first derivitive is 0 (for all x < 0) or undefined thats why solving a Neural network that uses RelU is gonna be difficult

The hyperbolic tangent activation,

$$
\tanh(z),
$$

does not have any differentiability problem. In fact,

$$
\frac{d}{dz}\tanh(z)=1-\tanh^2(z),
$$

so it is differentiable everywhere. However, a loss function involving tanh have infinite critical points where the first derivitive is 0 (for a point p where derivitive is 0 the point p + 2*pi*k will also have derivitive 0) or undefined thats why solving a Neural network that uses Tanh is gonna be difficult.


### Choice of SiLU and Absolute Value

To make the critical-point analysis more tractable, we investigate the use of SiLU and the absolute-value function as activation functions.

The SiLU activation is defined as

$$
\operatorname{SiLU}(z)
=
z\sigma(z)
=
\frac{z}{1+e^{-z}},
$$

where \(\sigma(z)\) denotes the logistic sigmoid function. Its derivative can be obtained explicitly using the product and chain rules:

$$
\frac{d}{dz}\operatorname{SiLU}(z)
=
\sigma(z)+z\sigma(z)(1-\sigma(z)).
$$

The absolute-value activation is defined as

$$
|z|,
$$

and its derivative, away from \(z=0\), is

$$
\frac{d}{dz}|z|
=
\operatorname{sign}(z)
=
\begin{cases}
-1, & z<0,\\
1, & z>0.
\end{cases}
$$

Although the absolute-value function is not differentiable at \(z=0\), its derivative is particularly simple on either side of this point. Therefore, its critical-point analysis can be performed by considering the corresponding regions separately.Its range is restricted to the min of 0 so solving the inside function |f(x)| gives us minimum point

The overall procedure is therefore:

1. **Define the model and loss function.**
2. **Differentiate the loss with respect to the model parameters.**
3. **Set the derivatives equal to zero to obtain the critical-point equations.**
4. **Solve for the candidate critical points.**
5. **Use the first- or second-derivative test to determine whether each critical point is a minimum.**
6. **Compare the resulting minima where necessary to identify the global minimum.**

This approach allows us to study analytical solutions for nonlinear models through the tools of differential calculus. Importantly, differentiability of an activation function by itself does not guarantee a closed-form solution. Instead, the objective is to select activation functions for which the resulting derivative and critical-point equations are sufficiently tractable to permit analytical investigation.


## Greedy Approch (Another approch)

We propose another approch as  greedy layer-wise approach for constructing the parameters of a multilayer neural network using the closed-form solution of ordinary least-squares regression. The central idea is to solve each linear transformation independently, apply the activation function to the resulting representation, and then treat the transformed representation as the input dataset for the next layer.

Let the original training dataset be represented by

$$
X\in\mathbb{R}^{m\times d_0},\qquad
Y\in\mathbb{R}^{m\times d_n}.
$$

For the first layer, the weights are obtained by solving the ordinary least-squares problem

$$
\min_{W_1}\|XW_1-Y\|_F^2.
$$

Its minimum-norm closed-form solution is

$$
\boxed{W_1=X^\dagger Y},
$$

where \(X^\dagger\) denotes the Moore--Penrose pseudoinverse. The corresponding linear prediction is then passed through the activation function \(\sigma\), producing a new representation

$$
\boxed{V_1=\sigma(XW_1)}.
$$

The matrix \(V_1\) is subsequently treated as the input matrix for the next regression problem. Thus, rather than jointly optimizing all network parameters, the second-layer weights are obtained by solving

$$
\min_{W_2}\|V_1W_2-Y\|_F^2,
$$

which gives

$$
\boxed{W_2=V_1^\dagger Y}.
$$

The resulting representation is again transformed through the activation function:

$$
\boxed{V_2=\sigma(V_1W_2)}.
$$

This process is repeated recursively. For layer \(k\), given the representation \(V_{k-1}\), the corresponding weight matrix is obtained as

$$
\boxed{W_k=V_{k-1}^\dagger Y},
$$

followed by

$$
\boxed{V_k=\sigma(V_{k-1}W_k)}.
$$

For a network with \(n\) layers and a linear output layer, the final layer is obtained as

$$
\boxed{W_n=V_{n-1}^\dagger Y},
$$

with the final prediction

$$
\boxed{\hat{Y}=V_{n-1}W_n}.
$$

Consequently, the complete greedy procedure can be summarized as

$$
\begin{aligned}
W_1 &= X^\dagger Y,\\
V_1 &= \sigma(XW_1),\\
W_2 &= V_1^\dagger Y,\\
V_2 &= \sigma(V_1W_2),\\
&\ \vdots\\
W_{n-1} &= V_{n-2}^\dagger Y,\\
V_{n-1} &= \sigma(V_{n-2}W_{n-1}),\\
W_n &= V_{n-1}^\dagger Y.
\end{aligned}
$$

The method therefore replaces the conventional joint optimization of all neural-network parameters with a sequence of closed-form least-squares problems. Each layer is fitted to the original target \(Y\), while the activation-transformed output of the preceding layer serves as the input to the subsequent regression.

It is important to distinguish this procedure from solving the global neural-network optimization problem. Although each individual regression is solved optimally in the least-squares sense, the resulting sequence of layer-wise solutions is not, in general, guaranteed to minimize the original end-to-end neural-network objective globally. The purpose of this formulation is therefore to investigate whether repeated closed-form regression combined with nonlinear transformations can provide an effective alternative to conventional joint optimization, and to characterize the conditions under which the resulting greedy solution coincides with, or approximates, the global optimum Therefore we experiment this Approch first in the following files.
